# Refactoring a Jupyter Notebook into a Python Script

This notebook shows how to move from **exploratory notebook code** to **reusable Python functions** using the King County Housing dataset.

This is a common real-world handoff:
- data scientists explore the data and test ideas in notebook cells,
- ML engineers identify the stable transformations,
- and those transformations are rewritten as importable functions.

The goal is not to remove exploration. The goal is to separate:
1. **one-off analysis** that helps us understand the data, and
2. **repeatable preprocessing logic** that should live in Python modules.

This notebook focuses on the reasoning behind the refactor. The companion notebook `king-county-data-preparation.ipynb` shows the function-extraction step by step.


> **Checkpoint:**
> You can identify which cells are exploratory and which should become reusable functions.

> **Common pitfalls:**
> - Copying notebook code into a script without simplifying it.
> - Keeping one-off analysis statements inside reusable preprocessing functions.
> - Forgetting to define a clear DataFrame-in/DataFrame-out contract.

> **Self-check:**
> Could another teammate import your refactored functions without needing notebook state or hidden variables?


## Visual Guide: Refactor Strategy

```mermaid
flowchart TD
    A["Exploratory notebook cells"]
    B["Identify stable cleaning rules"]
    C["Extract focused functions"]
    D["Move logic into a Python module"]
    E["Reuse outside the notebook"]

    A --> B --> C --> D --> E
```


In [ ]:
# pandas is used for tabular data exploration and cleanup.
import pandas as pd

In [ ]:
# Load the raw housing dataset from the shared repository data folder.
# We start in the notebook because this is still the exploratory phase.
df = pd.read_csv("../data/King_County_House_prices_dataset.csv")

# Display a sample so we can sanity-check the columns and raw values.
df.head(10)

## Data Preparation

Before we write reusable Python functions, we need to understand the dataset well enough to decide **which transformations are stable and worth keeping**.

The questions below are still part of the exploratory phase. They help us decide:
- what the data quality issues are,
- which rules are defensible,
- and which pieces of logic should later move into a Python module.


### Visual Guide: What Moves Out of the Notebook?

```mermaid
flowchart TD
    A["Notebook cell"]
    B{"Is the logic repeatable?"}
    C["Keep it in notebook"]
    D["Turn it into a function"]
    E["Move it into a module"]

    A --> B
    B -->|No| C
    B -->|Yes| D --> E
```


**1.**  How many house sales are in our dataset?


In [ ]:
# `shape` returns `(rows, columns)`.
# This is a quick way to understand the overall size of the dataset.
df.shape

Each row contains the details of one house sale --> **21597** house sales are included in our dataset.


**2.** What format is the data in (**int** = number, **float** = decimal, **str** = text)?
Which formats are surprising?


In [ ]:
# `.info()` gives us a schema summary: column names, non-null counts, and data types.
# This is one of the fastest ways to spot unexpected string columns or missing values.
df.info()

Our dataset contains 8 columns with decimal numbers (float), 11 columns with integers (int), and 2 columns with text (str/object).  

We expect all data on the size, number of rooms, the indices, and the information on the location to be recognized as numbers.  

The `date` is recognized as text, as well as the area of the basement (`sqft_basement`). This is surprising! A little further up, when we looked at the first 10 rows of our table, we could see why the values are not recognized as numbers: there is a `?` mixed in.  


**3.** Are there any missing values?


In [ ]:
# Count the missing values in each column.
# We use this to decide which columns need a cleanup strategy later.
df.isna().sum()

We have missing values in the columns `waterfront`, `view`, and `yr_renovated`, with the most missing values being in the `yr_renovated` column.  

Further on, we have to **think about how to deal with these missing values**.  


**4.** How many values are there for the individual variables?


In [ ]:
# `nunique()` shows how many distinct values each column contains.
# This helps us notice repeated IDs, low-cardinality flags, and suspicious variables.
df.nunique()

**Interesting findings:**  

**1.** There are 21420 different house IDs: this means that up to 177 houses may have been sold twice.  

**2.** There are 3622 expressions for the price of sold houses: many houses were sold for the same price.  

**3.** Although `grade` is an index from 1-13, there are only 11 different values included.  

**4.** The houses in our dataset were built in 116 different years.  


Next, we can display the **statistical distribution** of the individual columns. Here, too, you are sure to discover a few interesting insights.  


| Stat | Description |
|---|---|
| `count` | Indication of how many values are present in the column (NaN/missing values are not counted) |
| `mean` | Mean value of the data |
| `std` | Standard deviation of the data |
| `min` | The smallest value in the column |
| `25%` | 25% of the data is below this value |
| `50%` | 50% of the data is below this value (this value is called the **median**) |
| `75%` | 75% of the data is below this value |
| `max` | The largest value in the column |


In [ ]:
# `describe()` summarizes the numeric columns.
# Rounding makes the output easier to scan while we look for unusual values.
df.describe().round(2)

**Interesting findings:**  

**1.** On average, the houses in this dataset cost \$540,296. The most expensive house sold for \$7,700,000, the cheapest one for \$78,000.  

**2.** 50% of the houses have 3 or fewer bedrooms. The house with the most bedrooms has 33! (Is that possible?)  

**3.** On average, each house has 2.1 bathrooms.  

**4.** 75% of houses have 2 or fewer floors.  

**5.** Only 1% of the houses have a view of the seafront (since `waterfront` can only take the values 0 and 1, the **mean** can be interpreted as the percentage of houses with a view of the seafront).  

**6.** The oldest house in our dataset is from 1900, the newest one from 2015.  


Of course we can find many more insights from the table above, but we have already discovered a few inconsistencies.

These are strong candidates for the later refactor because they represent **repeatable data-cleaning rules**:
- missing values that need an explicit policy,
- `?` in `sqft_basement`, which should be numeric,
- and the suspicious house entry with 33 bedrooms.

The next few cells are intentionally notebook-style and one-off. Later, we will keep only the stable parts and move them into reusable functions.


**33 bedrooms**

If we look at the maximum number of bedrooms, we see a house with 33 bedrooms but less than 2 bathrooms.


In [ ]:
# Inspect the suspicious record directly before deciding on a cleaning rule.
# In exploratory work, this kind of one-off query is normal and often useful.
df.query("bedrooms == 33")

Since the ratio of 33 bedrooms to 2 bathrooms in 1620 sqft sounds very unlikely, we are removing this house from the record.


In [ ]:
# This is still one-off exploratory cleanup code.
# We remove the suspicious row now, then later rewrite the general logic as a reusable function.
df.drop(15856, axis=0, inplace=True)

**`?` in `sqft_basement`**

We have seen in the output of `.info()` that `sqft_basement` is presented as a string and not as a number. This is because there are some `?` in this column, even though basement area should be numeric.  

Because `sqft_living` and `sqft_above` already contain the information we need, it is safer to rebuild the entire `sqft_basement` feature from those two reliable source columns instead of trying to trust the messy raw column.  


In [ ]:
# Rebuild basement size directly from two columns that already contain the needed information.
# This is often safer than trusting a raw column that mixes numbers with placeholder strings.
df["sqft_basement"] = df["sqft_living"] - df["sqft_above"]

In [ ]:
# Inspect a few recalculated basement values.
df[["sqft_living", "sqft_above", "sqft_basement"]].head()

**Missing values**

Before choosing fill rules, we audit how many missing values are left and how large they are relative to the full dataset.


In [ ]:
# Build a small audit table with both counts and percentages.
# Percentages help us judge whether missingness is tiny, moderate, or substantial.
missing_values = df.isnull().sum().to_frame(name="count")
missing_values["percentage"] = (missing_values["count"] / df.shape[0] * 100).round(2)
missing_values.query("count != 0")

Only 3 features have missing values after rebuilding `sqft_basement` from the reliable source columns.  


In [ ]:
# Check the distribution of `view` before deciding how to fill its missing values.
# If one value is overwhelmingly common, a simple fill rule may be reasonable.
df["view"].value_counts()

The column `view` has only 0.29% of missing values and has 19421 out of 21596 times the value 0, so we will replace the missing values in this column with 0.  


In [ ]:
# Fill missing `view` values with 0.
# In this workflow, 0 represents "no view" and is also the dominant observed value.
df["view"] = df["view"].fillna(0)

In the column `waterfront` we see a similar distribution:


In [ ]:
# Inspect `waterfront` in the same way before choosing a fill value.
df["waterfront"].value_counts()

So also here we replace the NaNs with 0:

In [ ]:
# Fill missing `waterfront` values with 0.
# Again, this matches the dominant business meaning in this dataset.
df["waterfront"] = df["waterfront"].fillna(0)

In [ ]:
# Rebuild the missing-value summary after the fills above.
# This lets us confirm which columns still need attention.
missing_values = df.isnull().sum().to_frame(name="count")
missing_values["percentage"] = missing_values["count"] / df.shape[0] * 100
missing_values.query("count != 0")

Since 17.8% of the data is missing from `yr_renovated`, dropping the column outright would discard information and leaving it as-is would keep a messy signal.

A cleaner next step is to create one consolidated feature called `last_known_change`:
- if a house has no renovation year, fall back to `yr_built`,
- otherwise use the renovation year,
- then drop the two original source columns.

This is a good example of logic that belongs in a reusable function because it is deterministic and business-facing.


In [ ]:
# Create an empty list to collect the consolidated year values.
last_known_change = []

# Loop through the renovation-year column row by row.
for idx, yr_re in df["yr_renovated"].items():
    # Missing values or 0 mean "no known renovation",
    # so we fall back to the original build year.
    if str(yr_re) == "nan" or yr_re == 0.0:
        last_known_change.append(df["yr_built"][idx])
    else:
        # Otherwise, keep the renovation year as the last known change.
        last_known_change.append(int(yr_re))

In [ ]:
# Add the consolidated list back as a new feature.
df["last_known_change"] = last_known_change

In [ ]:
# Drop the source columns now that their information is captured in `last_known_change`.
df.drop("yr_renovated", axis=1, inplace=True)
df.drop("yr_built", axis=1, inplace=True)

We are **done with the data cleaning**.

At this stage, the notebook has done its job: it helped us inspect the data, justify a few rules, and test the transformations interactively.

The next step is to convert the **stable cleaning logic** into reusable Python functions. That extraction step is the bridge from notebook work to production-friendly code.


Now refactor the cleaned workflow into functions and save them in `src/king_county_refactoring`.

This notebook focuses on **what should be refactored** and **why**. The companion notebook `king-county-data-preparation.ipynb` walks through the individual function extractions in more detail.

**Exercise goal:** Move the stable cleaning logic into reusable Python functions that can be imported elsewhere.

@TODO:
1. Implement functions for outlier cleanup, basement conversion, renovation-year transformation, and missing-value filling.
2. Write your implementation in `src/king_county_refactoring/data_preparation.py`.
3. Keep each function DataFrame-in/DataFrame-out and copy-first.
4. Chain the functions and verify that `view` and `waterfront` have no missing values.

**Hints:**
- Keep exploratory display code in the notebook, not in the final functions.
- Each function should own one transformation responsibility.
- Validate intermediate results after each step while developing.

> **Checkpoint:**
> You can chain all four functions without manual intermediate fixes.

Starter file: `src/king_county_refactoring/data_preparation.py`  
Reference solution: `src/king_county_refactoring/data_preparation_solution.py`


### **Quiz**

1. **Why might we want to refactor a Jupyter Notebook into a Python script?**  
    [ ] To make the notebook look shorter  
    [ ] For reusability and reproducibility in production environments  
    [ ] Because Python scripts always run faster than notebooks  
    [ ] To avoid using pandas and scikit-learn  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** For reusability and reproducibility in production environments  
      
      **Description:** Python scripts can be version-controlled, reused, and integrated into production workflows more easily than Jupyter Notebooks.
    </details>

---

2. **Which of the following is a limitation of keeping code only inside a Jupyter Notebook?**  
    [ ] Code cannot run on a laptop  
    [ ] It is harder to reuse and test code in larger projects  
    [ ] Notebooks cannot contain visualizations  
    [ ] Variables cannot be defined  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It is harder to reuse and test code in larger projects  
      
      **Description:** While notebooks are great for exploration, scripts are better for modularity, testing, and integration with larger systems.
    </details>

---

3. **What is a common first step when refactoring a notebook into a script?**  
    [ ] Deleting all data analysis code  
    [ ] Moving reusable code (functions, preprocessing, etc.) into `.py` files  
    [ ] Changing all variables to uppercase  
    [ ] Removing markdown cells  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Moving reusable code (functions, preprocessing, etc.) into `.py` files  
      
      **Description:** By extracting reusable parts (e.g., feature engineering, data loading) into scripts, we make them easier to import and maintain.
    </details>

---

4. **What advantage does separating data preparation code into a script provide?**  
    [ ] It prevents missing values  
    [ ] It allows the same transformations to be applied consistently across different projects or runs  
    [ ] It makes the code harder to read  
    [ ] It avoids the need for testing  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It allows the same transformations to be applied consistently across different projects or runs  
      
      **Description:** A script ensures consistent and reproducible preprocessing, reducing errors from manual copy-pasting.
    </details>

---

5. **Who might be responsible for refactoring notebooks into production-ready scripts?**  
    [ ] Only software engineers  
    [ ] Only data scientists  
    [ ] Data scientists, software engineers, or machine learning engineers depending on the workflow  
    [ ] Only database administrators  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Data scientists, software engineers, or machine learning engineers depending on the workflow  
      
      **Description:** Both data scientists and ML engineers often collaborate — data scientists create prototypes, and ML engineers refactor for production use.
    </details>

---

6. **Which of the following best describes the relationship between data science and machine learning engineering in this context?**  
    [ ] They are completely separate and never overlap  
    [ ] They overlap, especially in areas like code refactoring and reproducibility  
    [ ] Data science replaces machine learning engineering  
    [ ] Machine learning engineering is only about writing models  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** They overlap, especially in areas like code refactoring and reproducibility  
      
      **Description:** Data science focuses on insights and prototyping, while ML engineering ensures production readiness. Their work intersects in areas like refactoring.
    </details>
